In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.orders_enriched
USING DELTA
AS
WITH typed_orders AS (
  SELECT
    o.order_id,
    o.customer_id,
    c.customer_state,
    o.order_status,
    o.purchase_month,
    CAST(o.order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
    CAST(o.order_delivered_customer_date AS DATE) AS actual_delivery_date,
    CAST(o.order_estimated_delivery_date AS DATE) AS estimated_delivery_date
  FROM olist.bronze.orders AS o
  LEFT JOIN olist.bronze.customers AS c
    ON o.customer_id = c.customer_id
)
SELECT *,
  CASE WHEN order_status = 'delivered'
    THEN DATEDIFF(actual_delivery_date, estimated_delivery_date)
  END AS days_vs_estimate
FROM typed_orders

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT assert_true(
  COUNT(*) > 0
  AND COUNT(*) = (SELECT COUNT(*) FROM olist.bronze.orders)
  AND COUNT(*) = COUNT(DISTINCT order_id)
  AND COUNT(*) = COUNT(customer_state),
  'Silver check failed: expected the Bronze row count, unique order IDs, and a state for every order.'
) AS silver_quality_check
FROM olist.silver.orders_enriched

silver_quality_check
null
